# Train on Kaggle (free GPU)

Runs the experiments in this repository on a Kaggle GPU and packs the results for download.

**One-time setup in the Kaggle editor (right-hand panel):**
1. *Settings → Accelerator →* **GPU T4 x2** (or P100).
2. *Settings →* **Internet on** (needed to fetch the code and the ImageNet weights).
3. *Add Input →* search **"New Plant Diseases Dataset"** by *vipoooool* → Add.

Then **Run All**. At the end, download `results.zip` from the *Output* panel and put it in
the repository (`ml/results/`), or hand it to whoever maintains the repo.

Nothing here edits numbers by hand. Every result comes from the files a run writes.

In [ ]:
# ---- settings -------------------------------------------------------------------------
# Where to get the code. A public GitHub URL, or leave None if the repository is attached
# as a Kaggle dataset (Add Input -> your upload); it is then found under /kaggle/input.
REPO_URL = None   # e.g. "https://github.com/<user>/<repo>.git"
BRANCH = "main"

# Which experiments to run (config files under ml/configs/). Each line is one run.
RUNS = [
    "configs/paper/cnn_fullepochs.yaml",
    "configs/paper/mobilenet_fullepochs.yaml",
    # "configs/paper/cnn_steps100.yaml",
    # "configs/paper/mobilenet_steps100.yaml",
]
EXTRA_OVERRIDES = []  # e.g. ["seed=1"]

In [ ]:
import os, subprocess, sys, glob
from pathlib import Path

WORK = Path("/kaggle/working")
if REPO_URL:
    repo = WORK / "repo"
    if not repo.exists():
        subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, REPO_URL, str(repo)], check=True)
else:
    hits = [Path(p).parent.parent for p in glob.glob("/kaggle/input/*/ml/pyproject.toml")]
    hits += [Path(p).parent.parent for p in glob.glob("/kaggle/input/*/*/ml/pyproject.toml")]
    assert hits, "Set REPO_URL or attach the repository as a Kaggle dataset."
    # inputs are read-only: copy the code (small) to the working directory
    repo = WORK / "repo"
    if not repo.exists():
        subprocess.run(["cp", "-r", str(hits[0]), str(repo)], check=True)
ML = repo / "ml"
os.chdir(ML)
# Kaggle already has TensorFlow, pandas, scikit-learn...; --no-deps keeps its TF build.
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "-e", "."], check=True)
print("code at", ML)

In [ ]:
# Find the NPD folder that contains train/ and valid/ with the 38 class folders.
cands = []
for p in glob.glob("/kaggle/input/**/train", recursive=True):
    root = Path(p).parent
    if (root / "valid").is_dir() and len(list(Path(p).iterdir())) == 38:
        cands.append(root)
assert cands, "Attach the 'New Plant Diseases Dataset' (vipoooool) as an input."
NPD_ROOT = sorted(cands, key=lambda r: str(r))[0]
print("NPD root:", NPD_ROOT)
for split in ("train", "valid"):
    n = sum(1 for _ in (NPD_ROOT / split).rglob("*.*"))
    print(f"  {split}: {n} files")  # expected 70295 / 17572

In [ ]:
import tensorflow as tf, keras
print("TensorFlow", tf.__version__, "| Keras", keras.__version__)
print("GPUs:", tf.config.list_physical_devices("GPU"))
assert tf.config.list_physical_devices("GPU"), "No GPU: enable one under Settings -> Accelerator."
print("RandomShear available:", hasattr(keras.layers, "RandomShear"))

In [ ]:
import time
for cfg in RUNS:
    t = time.time()
    cmd = [sys.executable, "-m", "leafdx.train.run", "--config", cfg,
           "--set", f"data.npd_root={NPD_ROOT}", f"output_dir={WORK / 'outputs'}", *EXTRA_OVERRIDES]
    print(">>>", " ".join(cmd), flush=True)
    subprocess.run(cmd, check=True)
    print(f"<<< {cfg} finished in {(time.time() - t) / 60:.1f} min", flush=True)

In [ ]:
# Pack the small result files (no model weights) for download.
subprocess.run([sys.executable, "-m", "leafdx.train.report", "collect", str(WORK / "outputs"),
                "--results", str(WORK / "results")], check=True)
subprocess.run(["zip", "-qr", str(WORK / "results.zip"), "results"], cwd=WORK, check=True)
print("download:", WORK / "results.zip")
for m in sorted((WORK / "results").glob("*/seed*/metrics.json")):
    import json; d = json.loads(m.read_text())
    print(f"{d['name']:32s} last-epoch val_accuracy={d['last_epoch']['val_accuracy']:.4f} "
          f"full-val accuracy={d['val_full']['accuracy']:.4f}")